# Session 3 - Lab: a variational autoencoder, and the dial that decides what it is for

**DAT09-06 Representation Learning · Friday 13 November 2026 · 1 h 20**

On Monday your autoencoder described products well and could not invent one:
32 random numbers decoded to a ghost. Today you give every product a **zone**
instead of a point, add the **penalty** that keeps the zones together, measure
what that buys, and find out what it costs.

### The words of the lecture, and the names your AI will use

| In the lecture | In papers and in code |
|---|---|
| the **zone** of a product: a centre μ and a width σ | the *posterior* |
| the **standard zone**: centre 0, width 1 | the *prior*, `N(0, I)` |
| the **penalty** | the *KL divergence*, `kl` |
| the **draw**: z = μ + σ × noise | the *reparameterization trick* |
| `logvar` | the log of the width squared: σ = `exp(0.5 * logvar)` |

### How this notebook is marked

| Marker | What it means |
|---|---|
| `# Given - read it, run it, move on.` | nothing to write. Run the cell. |
| ✏️ **YOUR TURN** | a boxed banner tells you **how many lines** and **what they do**. That is the only place you write code. |
| ❓ **Checkpoint** | **three questions** about *your* output. Answer **in writing**, in the cell below. Most only ask you to read what you just printed. |
| ⭐ **Going further** | optional, harder. Only if you are ahead: the next checkpoint never needs them. |

There are **7 cells to write**, none longer than six lines. Everything else is
given, including every training loop and every plot.

### Read this before you start

An AI will write all seven in two minutes. Let it - but **read what it writes**:
the most instructive bug of the day is one that an AI's first draft very often
contains, and it does not crash. It quietly switches the whole code off.

The questions are the lab. The code is the easy part.

### Six checkpoints

| | What you produce | ~ |
|---|---|---|
| **1** | From a point to a zone: the encoder's two outputs, and the draw | 15 min |
| **2** | The loss: the penalty checked on four zones, then a trained VAE | 20 min |
| **3** | Did it fill the empty space? Invented products, Monday vs today | 15 min |
| **4** | **The β dial** - four models, one table | 20 min |
| **5** | Numbers in use, numbers switched off | 10 min |
| **6** | **The challenge** - the best inventor that still describes well | open |

**Most of you will not reach checkpoint 6. That is deliberate.**

Run the cells **in order**.

---

## 0 · Setup

Four given cells: imports, data, Monday's autoencoder, and the judge.

In [ ]:
# Given - read it, run it, move on.
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))   # make `common` importable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from common.evaluation import evaluate_representation
from common.paths import PROCESSED, require
from common.plotting import CLASS_COLORS, PALETTE, use_course_style
from common.seeding import SEED, cpu_threads, set_seed

use_course_style()
set_seed(SEED)
cpu_threads()

print("torch", torch.__version__, "| device: cpu")

### 0.2 · The catalogue

Same split as Monday: 54,000 products to train on, 6,000 held out. Labels are
loaded, and used **only** to measure.

In [ ]:
# Given - read it, run it, move on.
blob = np.load(require(PROCESSED / "fmnist_train.npz",
                       hint="python data/prepare_data.py --fmnist"))
images, labels = blob["images"], blob["labels"]
class_names = list(blob["classes"])

x = images.reshape(len(images), -1).astype(np.float32) / 255.0   # (60000, 784)
x_train = torch.from_numpy(x[:54000].copy())
x_val = torch.from_numpy(x[54000:].copy())
y_train, y_val = labels[:54000], labels[54000:]

print(f"train {tuple(x_train.shape)}   val {tuple(x_val.shape)}")

### 0.3 · Monday's autoencoder, for comparison

The exact model from Monday: a code of 32 numbers, 8 epochs. A few seconds on
the reference machine; allow up to a minute and a half on a laptop.

In [ ]:
# Given - read it, run it, move on.
HIDDEN = 256


class Autoencoder(nn.Module):
    """Monday's model: 784 -> 256 -> d -> 256 -> 784."""

    def __init__(self, d, hidden=HIDDEN):
        super().__init__()
        self.d = d
        self.encoder = nn.Sequential(nn.Linear(784, hidden), nn.ReLU(), nn.Linear(hidden, d))
        self.decoder = nn.Sequential(nn.Linear(d, hidden), nn.ReLU(),
                                     nn.Linear(hidden, 784), nn.Sigmoid())

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z


def train_ae(model, data, epochs=8, batch=256):
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for _ in range(epochs):
        perm = torch.randperm(len(data))
        for i in range(0, len(data), batch):
            xb = data[perm[i:i + batch]]
            loss = F.mse_loss(model(xb)[0], xb)
            opt.zero_grad(); loss.backward(); opt.step()


set_seed(SEED)
ae32 = Autoencoder(32)
t0 = time.time()
train_ae(ae32, x_train)
with torch.no_grad():
    z_ae = ae32.encoder(x_val).numpy()
    ae_mse = float(((ae32(x_val)[0] - x_val) ** 2).mean())
ae_rep = evaluate_representation(z_ae, y_val, name="AE d=32", verbose=False)
print(f"trained in {time.time() - t0:.1f}s   val mse {ae_mse:.5f}   "
      f"knn {ae_rep.knn_accuracy:.3f}   probe {ae_rep.probe_accuracy:.3f}")

### 0.4 · The judge

Today we invent pictures, so we need a way to score a picture without a human
looking at it. The **judge** is a classifier: a network trained on the 54,000
labelled photos to name the kind of product (T-shirt, trouser, sneaker… ten
kinds), with a confidence for each kind.

**The rule:** a picture is **recognised** when the judge is **more than 90 %
sure** of one kind. `recognised(pictures)` returns the share of pictures that
pass. It is an imperfect ruler - even real photos do not all pass - but it is
the same ruler for every model.

In [ ]:
# Given - read it, run it, move on.
set_seed(SEED)
judge = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 10))
opt = torch.optim.Adam(judge.parameters(), lr=1e-3)
yt = torch.from_numpy(y_train).long()
for _ in range(3):
    perm = torch.randperm(len(x_train))
    for i in range(0, len(x_train), 256):
        idx = perm[i:i + 256]
        loss = F.cross_entropy(judge(x_train[idx]), yt[idx])
        opt.zero_grad(); loss.backward(); opt.step()
judge.eval()


@torch.no_grad()
def recognised(imgs, threshold=0.9):
    """Share of pictures for which the judge is more than 90 % sure of one kind."""
    p = judge(torch.as_tensor(imgs)).softmax(1).max(1).values
    return float((p > threshold).float().mean())


with torch.no_grad():
    acc = (judge(x_val).argmax(1).numpy() == y_val).mean()
print(f"the judge names the right kind for {acc:.3f} of real held-out photos")
print(f"real held-out photos recognised: {recognised(x_val):.3f}   <- the ceiling")

In [ ]:
# Given - read it, run it, move on.
def show_row(imgs, title=None, n=8, labels=None):
    """Plot n images side by side."""
    fig, axes = plt.subplots(1, n, figsize=(n * 1.1, 1.5))
    for c, ax in enumerate(axes):
        ax.imshow(1 - np.asarray(imgs[c]).reshape(28, 28), cmap="gray", vmin=0, vmax=1)
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        if labels is not None:
            ax.set_title(labels[c], fontsize=9)
    if title:
        fig.suptitle(title, y=1.15 if labels is not None else 1.08)
    plt.show()

---

# Checkpoint 1 - From a point to a zone

**~15 minutes.** One cell to write, two lines.

### 1.1 · The encoder now has two outputs

Monday's encoder ended in one layer of 32 numbers: the code. The VAE's encoder
ends in **two** layers of 32 numbers: the **centre** μ of the product's zone,
and its **width** σ.

One oddity, which every implementation shares: the second layer does not output
the width itself but `logvar`, the **log of the width squared**. That way the
layer may output any number, and the width σ = `exp(0.5 * logvar)` is always
positive.

The decoder is Monday's, unchanged.

In [ ]:
# Given - read it, run it, move on.
class VAE(nn.Module):
    """784 -> 256 -> (mu, logvar), each of size d;  z -> 256 -> 784."""

    def __init__(self, d, hidden=HIDDEN):
        super().__init__()
        self.d = d
        self.body = nn.Sequential(nn.Linear(784, hidden), nn.ReLU())
        self.mu = nn.Linear(hidden, d)          # the centres
        self.logvar = nn.Linear(hidden, d)      # the widths, as log(width squared)
        self.decoder = nn.Sequential(nn.Linear(d, hidden), nn.ReLU(),
                                     nn.Linear(hidden, 784), nn.Sigmoid())

    def encode(self, x):
        h = self.body(x)
        return self.mu(h), self.logvar(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = draw_z(mu, logvar)                  # a point in the zone: you write it below
        return self.decoder(z), mu, logvar

### 1.2 ✏️ · The draw

Pick a point in the zone, exactly as on the slide:

**z = μ + σ × noise**

where the noise is drawn from the standard bell. `torch.randn_like(t)` gives
such noise, with the shape of `t`.

The check below uses the slide's example: centre 2.0, width 0.5.

> **Watch out.** σ = `exp(0.5 * logvar)` - the 0.5 is easy to forget, and
> nothing crashes if you do.

*(Papers call this line the reparameterization trick.)*

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about 2 lines
#     sigma = exp(0.5 * logvar)
#     return mu + sigma * (noise shaped like sigma)
# ──────────────────────────────────────────────────────────────────────
def draw_z(mu, logvar):
    """One point drawn in each zone: centre + width x noise."""
    ...          # <- replace this line


z = draw_z(torch.full((100000, 1), 2.0), torch.full((100000, 1), np.log(0.5 ** 2)))
print(f"100,000 draws: average {z.mean():.3f} (expect 2.0)   spread {z.std():.3f} (expect 0.5)")

### 1.3 ▶ · A demonstration: why the draw is written this way

**Nothing to write, and no need to read the code line by line.** Run the cell
and read the three lines it prints.

Monday: training sends each number of the network a message - the *gradient* -
that says "move this way to lower the loss". The cell draws a point in a zone
in three ways, and prints the message that reaches the centre μ:

- `.sample()` - PyTorch throws the dice **inside** the network;
- `.rsample()` - PyTorch's own version of the line you just wrote;
- yours.

`None` means: **no message**. An encoder that gets no message cannot learn
where to put its zones. That is the whole reason for writing the draw as
"centre + width × noise".

In [ ]:
# Given - read it, run it, move on.
def message_reaching_mu(draw):
    mu = torch.tensor([0.5], requires_grad=True)
    logvar = torch.tensor([0.0], requires_grad=True)
    z = draw(mu, logvar)
    loss = (z - 3.0) ** 2           # any loss that depends on z
    if z.requires_grad:
        loss.backward()
    return mu.grad


dice = lambda mu, lv: torch.distributions.Normal(mu, torch.exp(0.5 * lv)).sample()
builtin = lambda mu, lv: torch.distributions.Normal(mu, torch.exp(0.5 * lv)).rsample()

torch.manual_seed(0); print("with .sample()  : message to mu =", message_reaching_mu(dice))
torch.manual_seed(0); print("with .rsample() : message to mu =", message_reaching_mu(builtin))
torch.manual_seed(0); print("with yours      : message to mu =", message_reaching_mu(draw_z))

### ❓ Checkpoint 1

1. In 1.2, what average and what spread do your 100,000 draws have? Now remove
   the `0.5` from your line and run it again: what spread do you get? *(Then put
   the `0.5` back.)*
2. In 1.3, which of the three ways sends **no message** to μ? What do the other
   two print?
3. `model.encode(x)` returns the centre and the width; `model(x)` also draws a
   point. Call each twice on the same photo: which one gives the same answer
   both times? Which would you store as the product's description for search?
   *(You can answer without running anything.)*

*Your answers:*

1.
2.
3.

---

# Checkpoint 2 - The loss: check it, then train

**~20 minutes.** Two cells to write.

**loss = reconstruction error + β × penalty**

- *reconstruction error*: redraw the product, starting from a point drawn in its zone;
- *penalty*: keep every zone close to the standard one, centre 0 and width 1.

### 2.1 ✏️ · The penalty is one line

For one number of the code, with centre μ and width σ:

`0.5 * (μ² + σ² − 1 − log(σ²))`

then **add up over the 32 numbers** of the code. In code, σ² is `logvar.exp()`
and log(σ²) is simply `logvar`.

Return **one penalty per product**: add over the numbers of the code, do not
average over the batch here.

*(Papers call the penalty the KL divergence.)*

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about 1 line
#     the line above, with sigma squared = logvar.exp(),
#     added up over dim=1
# ──────────────────────────────────────────────────────────────────────
def penalty(mu, logvar):
    """How far each product's zone is from the standard one. Shape (n,)."""
    ...          # <- replace this line

### 2.2 ▶ · Check it on the four zones of the lecture

The slide showed four zones and what each one pays. Your line should give the
same four numbers.

In [ ]:
# Given - read it, run it, move on.
slide = [(0.0, 1.0, 0.00), (2.0, 1.0, 2.00), (0.0, 0.2, 1.13), (3.0, 0.2, 5.63)]
for centre, width, expected in slide:
    mine = penalty(torch.tensor([[centre]]), torch.tensor([[2 * np.log(width)]])).item()
    print(f"centre {centre:g}, width {width:g}   yours {mine:5.2f}   slide {expected:5.2f}")

# a code made of these four numbers pays the four penalties, added up
mu_t = torch.tensor([[0.0, 2.0, 0.0, 3.0]])
logvar_t = 2 * torch.log(torch.tensor([[1.0, 1.0, 0.2, 0.2]]))
print(f"one product with these four numbers: {penalty(mu_t, logvar_t).item():.2f}   (expect 8.76)")

### 2.3 ✏️ · The loss

Reconstruction error: we use the **binary cross-entropy** - another way than
Monday's squared error to measure how far the redrawn picture is from the
photo, and the usual one for VAEs. It must be **added up over the 784 pixels**,
then averaged over the batch.

> **The trap.** `F.binary_cross_entropy(x_hat, x)` and `nn.MSELoss()` both
> **average** over the pixels by default. The reconstruction error becomes 784
> times smaller, and the penalty wins. Use `reduction="sum"`, then divide by
> the batch size yourself. You will see what the default does in 2.6.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about 3 lines
#     run the model: it returns the redrawn pictures, mu and logvar
#     rec: binary cross-entropy ADDED UP over pixels, divided by len(x)
#     kl:  penalty(...), averaged over the batch
# ──────────────────────────────────────────────────────────────────────
def vae_loss(model, x, beta=1.0):
    """Returns (loss, reconstruction error, penalty), each averaged over the batch."""
    ...          # <- replace this line
    return rec + beta * kl, rec, kl

### 2.4 · The training loop

Given, and you will not touch it again. It is Monday's loop with `vae_loss` in
place of the squared error, and it records both halves of the loss.

In [ ]:
# Given - read it, run it, move on.
def train_vae(model, data, *, beta=1.0, epochs=8, batch=256, lr=1e-3, verbose=True):
    """Train a VAE; return the per-epoch (reconstruction error, penalty) history."""
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(len(data))
        rec_sum = kl_sum = 0.0
        for i in range(0, len(data), batch):
            xb = data[perm[i:i + batch]]
            loss, rec, kl = vae_loss(model, xb, beta)
            opt.zero_grad(); loss.backward(); opt.step()
            rec_sum += rec.item() * len(xb)
            kl_sum += kl.item() * len(xb)
        history.append((rec_sum / len(data), kl_sum / len(data)))
        if verbose:
            r, k = history[-1]
            print(f"  epoch {epoch + 1:>2}/{epochs}   reconstruction error = {r:6.1f}   "
                  f"penalty = {k:5.2f}   loss = {r + k:6.1f}")
    return history

### 2.5 ▶ · Train one: a code of 32 numbers, β = 1

A few seconds on the reference machine.

In [ ]:
# Given - read it, run it, move on.
set_seed(SEED)
vae = VAE(32)
t0 = time.time()
hist = train_vae(vae, x_train, beta=1.0)
print(f"trained in {time.time() - t0:.1f}s")

fig, ax = plt.subplots(figsize=(6.4, 3.2))
ep = range(1, len(hist) + 1)
ax.plot(ep, [h[0] for h in hist], marker="o", color=PALETTE["accent"])
ax.set_ylabel("reconstruction error", color=PALETTE["accent"]); ax.set_xlabel("epoch")
ax2 = ax.twinx(); ax2.grid(False)
ax2.plot(ep, [h[1] for h in hist], marker="s", color=PALETTE["teal"])
ax2.set_ylabel("penalty", color=PALETTE["teal"])
ax.set_title("Checkpoint 2 - the two halves of the loss during training")
plt.show()

In [ ]:
# Given - read it, run it, move on.
@torch.no_grad()
def mu_of(model, x):
    """The centres of the zones: the VAE's description of each product. numpy (n, d)."""
    model.eval()
    return model.encode(x)[0].numpy()


@torch.no_grad()
def penalty_per_number(model, x):
    """Average penalty paid by each of the d numbers of the code. numpy (d,)."""
    model.eval()
    mu, logvar = model.encode(x)
    return (0.5 * (mu ** 2 + logvar.exp() - 1.0 - logvar)).mean(0).numpy()


@torch.no_grad()
def decode(model, z):
    model.eval()
    return model.decoder(torch.as_tensor(z, dtype=torch.float32)).numpy()


IN_USE = 0.1   # a number of the code is "in use" if it pays more than 0.1 of penalty on average

mu_val = mu_of(vae, x_val)
vae_rep = evaluate_representation(mu_val, y_val, name="VAE d=32 b=1", verbose=False)
vae_mse = float(((decode(vae, mu_val) - x_val.numpy()) ** 2).mean())
pn = penalty_per_number(vae, x_val)
print(f"VAE     val mse {vae_mse:.5f}   knn {vae_rep.knn_accuracy:.3f}   "
      f"probe {vae_rep.probe_accuracy:.3f}   numbers in use {(pn > IN_USE).sum()} / 32")
print(f"Monday  val mse {ae_mse:.5f}   knn {ae_rep.knn_accuracy:.3f}   "
      f"probe {ae_rep.probe_accuracy:.3f}")

### 2.6 ▶ · The trap, on purpose

The same VAE, trained with the default **mean** of `mse_loss` - the line an
AI's first draft very often contains. Three epochs are enough to see it.

In [ ]:
# Given - read it, run it, move on.
def vae_loss_averaged(model, x, beta=1.0):
    x_hat, mu, logvar = model(x)
    rec = F.mse_loss(x_hat, x)                 # averaged over the 784 pixels: the trap
    kl = penalty(mu, logvar).mean()
    return rec + beta * kl, rec, kl


correct_loss = vae_loss
vae_loss = vae_loss_averaged                   # swap it in, train, swap it back
set_seed(SEED)
trapped = VAE(32)
train_vae(trapped, x_train, beta=1.0, epochs=3, verbose=False)
vae_loss = correct_loss

pn_trap = penalty_per_number(trapped, x_val)
print(f"numbers in use : {(pn_trap > IN_USE).sum()} / 32   (correct loss: {(pn > IN_USE).sum()})")
print(f"total penalty  : {pn_trap.sum():.2f}   (correct loss: {pn.sum():.2f})")
show_row(decode(trapped, mu_of(trapped, x_val[:8])), "redrawn with the trapped loss")
show_row(x_val[:8].numpy(), "the originals")

### ❓ Checkpoint 2

1. Do your four penalties match the slide? Which costs more: moving the centre
   from 0 to 2, or shrinking the width from 1 to 0.2?
2. Compare the VAE's `val mse`, `knn` and `probe` with Monday's autoencoder
   (both printed in 2.5). Did the pictures get blurrier? Did the description of
   the products get worse?
3. In 2.6, how many of the 32 numbers stay in use with the trapped loss? What
   do the redrawn pictures look like?

**⭐ Going further - optional**

4. Explain **why** averaging over the pixels does this. *(Look at the last line
   of the training in 2.5: how big is the reconstruction error next to the
   penalty? What happens to that comparison when the error is divided by
   784?)*

*Your answers:*

1.
2.
3.
⭐ 4.

---

# Checkpoint 3 - Did it fill the empty space?

**~15 minutes.** One cell to write, two lines.

### 3.1 ▶ · Monday's autoencoder, as fairly as we can

To invent with Monday's decoder we must guess where its codes live. The fairest
guess available: for each of the 32 numbers, draw from a bell with the same
centre and the same spread as the real codes. Nobody told the autoencoder where
its codes should be, so a guess is all we have.

In [ ]:
# Given - read it, run it, move on.
rng = np.random.default_rng(SEED)
z_guess = rng.normal(z_ae.mean(0), z_ae.std(0), size=(1000, 32)).astype(np.float32)
with torch.no_grad():
    ae_samples = ae32.decoder(torch.from_numpy(z_guess)).numpy()
print(f"Monday's decoder, random numbers   recognised: {recognised(ae_samples):.3f}")
show_row(ae_samples, "Monday's decoder, random numbers drawn where the codes are")

### 3.2 ✏️ · The VAE, with no guessing

The VAE was told where its codes should live: around the **standard zone**,
centre 0 and width 1. So draw 1,000 codes of 32 numbers from the standard bell,
and decode them. This is the two-step recipe of the lecture.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about 2 lines
#     z_new: 1000 codes of 32 numbers from the standard bell (torch.randn)
#     vae_samples: decode them with the VAE
# ──────────────────────────────────────────────────────────────────────
set_seed(SEED + 9)
z_new = ...            # <- replace this line
vae_samples = ...      # <- replace this line

In [ ]:
# Given - read it, run it, move on.
print(f"VAE decoder, random numbers   recognised: {recognised(vae_samples):.3f}")
print(f"real photos                  recognised: {recognised(x_val):.3f}")
show_row(vae_samples, "VAE decoder, numbers drawn from the standard bell")

### 3.3 ▶ · Look at the two maps, with a code of 2 numbers

With 32 numbers per product there is nothing to look at. So we train two small
models whose code has only **2 numbers**, and draw one dot per product. Colours
show the kind of product (the labels are only used for the colour).

In [ ]:
# Given - read it, run it, move on.
set_seed(SEED); ae2 = Autoencoder(2); train_ae(ae2, x_train)
set_seed(SEED); vae2 = VAE(2); train_vae(vae2, x_train, beta=1.0, verbose=False)
with torch.no_grad():
    z2_ae = ae2.encoder(x_val).numpy()
z2_vae = mu_of(vae2, x_val)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
for ax, z, title in ((axes[0], z2_ae, "Monday's autoencoder: the codes"),
                     (axes[1], z2_vae, "VAE: the centres of the zones")):
    for c in range(10):
        s = y_val == c
        ax.scatter(z[s, 0], z[s, 1], s=3, alpha=0.6, color=CLASS_COLORS[c], label=class_names[c])
    ax.set_title(title); ax.set_aspect("equal", adjustable="datalim")
for r in (1, 2, 3):   # 1, 2 and 3 widths of the standard bell
    axes[1].add_patch(plt.Circle((0, 0), r, fill=False, ls="--", color=PALETTE["ink_soft"]))
axes[1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5), markerscale=4, fontsize=8)
plt.show()
print(f"Monday's codes go from {z2_ae.min(0).round(1)} to {z2_ae.max(0).round(1)}")
print(f"spread of the VAE's centres: {z2_vae.std(0).round(2)}")

### ❓ Checkpoint 3

1. Write down the three scores: real photos, Monday's decoder, the VAE. By what
   factor did the VAE improve on Monday? Is it close to the real photos?
2. Describe the VAE's pictures in words. Are they sharp? Would you show them to
   a customer?
3. Look at the two maps. From where to where do Monday's codes go? Which of the
   dashed circles (1, 2 and 3 widths of the standard bell) do the VAE's centres
   fill? Is that what the penalty asked for?

**⭐ Going further - optional**

4. To invent with Monday's decoder you had to *guess* where its codes are, with
   "a bell around the real codes". Use Monday's map to say why that is a poor
   guess.

*Your answers:*

1.
2.
3.
⭐ 4.

---

# Checkpoint 4 - The β dial

**~20 minutes. The most important part of the lab.** One cell to write.

β multiplies the penalty. β = 1 is the standard recipe; anything else is a
choice. You will train four VAEs that differ **only** in β and put everything
you know how to measure in one table.

### 4.1 ✏️ · One row of the table

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about 4 lines
#     train m with train_vae at this beta (verbose=False)
#     mu  = the centres of the zones of x_val (mu_of)
#     rep = evaluate_representation(mu, y_val, ...)
#     pn  = the penalty paid by each number, on x_val (penalty_per_number)
# ──────────────────────────────────────────────────────────────────────
def dial_row(beta):
    """Train a 32-number VAE at this beta, measure it. Returns (row, model)."""
    set_seed(SEED)
    m = VAE(32)
    ...          # <- replace this line
    row = {"beta": beta,
           "val mse": float(((decode(m, mu) - x_val.numpy()) ** 2).mean()),
           "penalty": float(pn.sum()), "numbers in use": int((pn > IN_USE).sum()),
           "knn": rep.knn_accuracy, "probe": rep.probe_accuracy,
           "recognised": recognised(decode(m, torch.randn(1000, 32)))}
    return row, m


row, _ = dial_row(1.0)
print(row)          # sanity check: close to your checkpoint 2 numbers

### 4.2 ▶ · Four models, well under a minute

In [ ]:
# Given - read it, run it, move on.
rows, dial_models = [], {}
for beta in (0.5, 1.0, 4.0, 20.0):
    row, dial_models[beta] = dial_row(beta)
    rows.append(row)

rows.append({"beta": "Monday's AE", "val mse": ae_mse, "penalty": None, "numbers in use": None,
             "knn": ae_rep.knn_accuracy, "probe": ae_rep.probe_accuracy,
             "recognised": recognised(ae_samples)})
table = pd.DataFrame(rows).set_index("beta")
table.round(4)

In [ ]:
# Given - read it, run it, move on.
ids = [0, 3, 7, 12, 18, 25]
fig, axes = plt.subplots(len(ids), 5, figsize=(5.6, 6.6))
cols = [("original", x_val[ids].numpy())] + [
    (f"beta={b:g}", decode(dial_models[b], mu_of(dial_models[b], x_val[ids])))
    for b in (0.5, 1.0, 4.0, 20.0)]
for c, (name, imgs) in enumerate(cols):
    for r in range(len(ids)):
        axes[r][c].imshow(1 - imgs[r].reshape(28, 28), cmap="gray", vmin=0, vmax=1)
        axes[r][c].set_xticks([]); axes[r][c].set_yticks([]); axes[r][c].grid(False)
    axes[0][c].set_title(name, fontsize=9)
plt.show()

### ❓ Checkpoint 4 - the questions that matter most today

`val mse` is the reconstruction error (lower is sharper). `knn` and `probe` are
Monday's numbers: they judge the **description**. `recognised` judges the
**invention**.

1. As β goes from 0.5 to 20, what happens to `val mse`? To `numbers in use`?
2. Which β gives the best `knn`, and is it better or worse than Monday's
   autoencoder? Which β gives the best `recognised`? Is it the same β?
3. You are building the assistant's **search** (find the products closest to a
   query): which β do you ship? Now you are building **"show me this dress, but
   longer"** (move in the space of codes, decode): which one? Give two numbers
   from your table for each choice.

**⭐ Going further - optional**

4. At β = 20 the invented pictures are recognised *less* often than at β = 4,
   although the zones are closer to the standard one. Look at the grid of
   redrawn products and explain.
5. At β = 20, `knn` collapses but `probe` drops much less. With only two
   numbers in use, why would a linear probe hold up better than a vote among
   nearest neighbours?

*Your answers:*

1.
2.
3.
⭐ 4.
⭐ 5.

---

# Checkpoint 5 - Numbers in use, numbers switched off

**~10 minutes.** One cell to write.

### 5.1 ▶ · Which of the 32 numbers carry something

One bar per number of the code: the penalty it pays on average. A number that
pays (almost) nothing has the standard zone for every product: it is switched
off.

In [ ]:
# Given - read it, run it, move on.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), sharey=True)
for ax, b in zip(axes, (1.0, 4.0)):
    k = np.sort(penalty_per_number(dial_models[b], x_val))[::-1]
    ax.bar(range(1, 33), k, color=[PALETTE["teal"] if v > IN_USE else PALETTE["rule"] for v in k])
    ax.axhline(IN_USE, ls="--", color=PALETTE["accent"], lw=1)
    ax.set_title(f"beta = {b:g}: {(k > IN_USE).sum()} of the 32 numbers are in use")
    ax.set_xlabel("the 32 numbers of the code, sorted")
axes[0].set_ylabel("average penalty paid")
plt.show()

### 5.2 ✏️ · Move one number, leave the others alone

Take the centre μ of one product, and replace **one** of its 32 numbers by the
values −3, −2, …, 3, leaving the other 31 alone. Decode the seven codes. Do it
for the **most used** number and for a **switched-off** one, in the β = 4
model.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about 1 line
#     overwrite column `number` of z with `steps`
# ──────────────────────────────────────────────────────────────────────
m4 = dial_models[4.0]
order = np.argsort(penalty_per_number(m4, x_val))   # switched-off first, most used last
base = mu_of(m4, x_val[3:4]).repeat(7, axis=0)      # one product, 7 copies
steps = np.arange(-3, 4, dtype=np.float32)


def walk(number):
    z = base.copy()
    ...          # <- replace this line
    return decode(m4, z)

In [ ]:
# Given - read it, run it, move on.
labels = [f"{s:+.0f}" for s in steps]
show_row(walk(order[-1]), f"the most used number (n° {order[-1]})", n=7, labels=labels)
show_row(walk(order[-2]), f"the second most used (n° {order[-2]})", n=7, labels=labels)
show_row(walk(order[0]), f"a switched-off number (n° {order[0]})", n=7, labels=labels)

### ❓ Checkpoint 5

1. What changes along the most used number? Along the second? Give each a name
   in plain words ("width", "darkness", "shoe or shirt"…).
2. What changes along the switched-off one? On the slide, only one zone pays no
   penalty: so what are the centre and the width of that number's zone, for
   every product?
3. Your search system stores all 32 numbers of μ at β = 4. How many of them are
   doing any work? What would you change in production?

*Your answers:*

1.
2.
3.

---

# Checkpoint 6 - The challenge

**Open-ended. Most of you will not get here, and that is fine.**

> Build **one** VAE that is good at both jobs: get the highest **`recognised`**
> score for products invented from the standard bell, while keeping the
> **`knn` of the centres at 0.80 or more**, with a code of **at most 32
> numbers** and **no more than 60 seconds of training**.

The reference is the plain β = 1 model: about **23 %** recognised at **0.82**
`knn`.

| Lever | Try |
|---|---|
| `beta` | between 1 and 4 - where exactly does `knn` cross 0.80? |
| `warmup` | start at β = 0 and raise it to its value over the first *n* epochs |
| `free_bits` | the free allowance: a number is not charged for penalty below this value |
| `d`, `hidden`, `epochs` | size of the code, size of the networks, length of training |

**The rules.** Time only the training. Keep a row for **every** configuration,
including the losers. One change at a time.

### 6.1 ▶ · The harness, given

In [ ]:
# Given - read it, run it, move on.
results = []


def attempt(name, *, d=32, beta=1.0, epochs=8, warmup=0, free_bits=0.0, hidden=HIDDEN):
    """Train under the clock, with optional warm-up and free allowance; record the row."""
    set_seed(SEED)
    m = VAE(d, hidden)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    steps_per_epoch = (len(x_train) + 255) // 256
    step, t0 = 0, time.time()
    for _ in range(epochs):
        m.train()
        perm = torch.randperm(len(x_train))
        for i in range(0, len(x_train), 256):
            xb = x_train[perm[i:i + 256]]
            x_hat, mu, logvar = m(xb)
            rec = F.binary_cross_entropy(x_hat, xb, reduction="sum") / len(xb)
            pen = (0.5 * (mu ** 2 + logvar.exp() - 1.0 - logvar)).mean(0)   # one per number
            kl = torch.clamp(pen, min=free_bits).sum() if free_bits else pen.sum()
            b = beta * min(1.0, step / (warmup * steps_per_epoch)) if warmup else beta
            loss = rec + b * kl
            opt.zero_grad(); loss.backward(); opt.step()
            step += 1
    secs = time.time() - t0
    rep = evaluate_representation(mu_of(m, x_val), y_val, name=name, verbose=False)
    rate = recognised(decode(m, torch.randn(1000, d)))
    ok = "ok " if rep.knn_accuracy >= 0.80 else "OUT"
    results.append({"config": name, "seconds": round(secs, 1), "knn": rep.knn_accuracy,
                    "recognised": rate, "valid": rep.knn_accuracy >= 0.80})
    print(f"  {ok} {name:<32s} {secs:5.1f}s   knn={rep.knn_accuracy:.4f}   recognised={rate:.3f}")
    return m


_ = attempt("reference: beta=1")

### 6.2 ✏️ · Your attempts

One `attempt(...)` per configuration. Change **one thing** at a time.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about as many lines as you have ideas
#     one attempt(...) per configuration, one change at a time
# ──────────────────────────────────────────────────────────────────────
# _ = attempt("beta=2", beta=2.0)
# _ = attempt("beta=2, warmup 2", beta=2.0, warmup=2)
# ... keep going, and keep the losers

In [ ]:
# Given - read it, run it, move on.
pd.DataFrame(results).sort_values(["valid", "recognised"], ascending=False)

### ❓ Checkpoint 6

1. Your best valid configuration, its `recognised` score and its `knn`.
2. **Which single lever mattered most?** Which two rows of your table prove it?
3. Did warm-up or the free allowance help *here*? Give the scores.

**⭐ Going further - optional**

4. Warm-up and the free allowance were invented for a decoder strong enough to
   ignore its code entirely. Is ours that strong? *(Checkpoint 5 is
   evidence.)*

*Your answers:*

1.
2.
3.
⭐ 4.

---

## What you built today

- An encoder that gives each product a **zone** - a centre and a width - and
  the one line that draws a point in it.
- The VAE loss as code: a reconstruction error added up over the pixels, plus
  a one-line **penalty**, checked against the four zones of the lecture.
- A measured answer to Monday's problem: products invented by the VAE are
  recognised about **twice** as often as Monday's - and are still far from real
  photos.
- A dial, not a model: the β that describes best is not the one that invents
  best, and too much of it switches the code off, one number at a time.

**Monday:** we stop redrawing pixels altogether. Contrastive learning tells the
model which pictures should be close, and lets it work out the rest.